In [ ]:
load("visualize.sage")
load("surfaces.sage")
load("curves.sage")

We consider a general surface $S$ in $\mathbb P^3$ defined by a polynomial $F \in \mathbb Q[x_0,x_1,x_2,x_3]$ of degree $d$.

In [ ]:
K = QQ #GF(9973)
R.<x0,x1,x2,x3> = K['x0,x1,x2,x3']
degree = 4
S_F = general_surface(degree)
F = S_F.gens()[0] 
print F

In [ ]:
plot_surf(F) # requires Surfer - imaginary

### Event surfaces
We compute the flecnodal (4), cusp crossing (3+2), and tritangent (2+2+2) as curves in ${\rm Gr}(1,\mathbb P^3)$.

In [ ]:
R1.<x0, x1, x2, x3, p01, p02, p03, p12, p13, p23, t> = K[]

event_type = '4' #change type

P = vector(R1,[1,0,-p12,-p13])
Q = vector(R1,[0, 1, p02, p03])
F1 = S_F.change_ring(R1)
poly = (F1.subs(x0=((P + t * Q)[0]), x1=((P + t * Q)[1]), 
                x2=((P + t * Q)[2]), x3 =((P + t * Q)[3]))).gens()[0]
c=poly.polynomial(t).coefficients()
disc=load("discriminants/disc%d_%s" %(degree,event_type))
v = disc.ring().gens()
disc = disc.change_ring(K[v])

sub_dict = dict(zip((disc.ring()).gens(),c))
event_surface = (disc.subs(sub_dict)).change_ring(R1)
event_surface = ideal(event_surface)+ideal(1*p23-p02*p13+p03*p12) #.change_ring(R2)
event_surface = ideal(event_surface.groebner_basis())
event_surface = (event_surface.homogenize(p01)).change_ring(K[p01, p02, p03, p12, p13, p23])

print "Dimension: ", event_surface.dimension()-1
print "Degree: ", event_surface.hilbert_polynomial().coefficients()[1]
#print event_surface

We next compute the parabolic surface in ${\rm Gr}(1,\mathbb P^3)$ (works for degree 3).

In [ ]:
R1.<x0, x1, x2, x3, y0, y1, y2, y3, p01, p02, p03, p12, p13, p23> = K[]

F1 = (ideal(F).change_ring(R1)).gens()[0]
H = jacobian(jacobian(F1, (x0,x1, x2, x3)), (x0, x1, x2, x3))
X = matrix(R1,[[x0], [x1], [x2], [x3]])
Y = matrix(R1,[[y0], [y1], [y2], [y3]])

XY = X.augment(Y)
XYmin = XY.minors(2)
I = ideal(F1) + ideal(H.determinant()) + ideal((H*Y).list()) + ideal(p01-XYmin[0],
    p02-XYmin[1],p03-XYmin[2],p12-XYmin[3],p13-XYmin[4],p23-XYmin[5])

parabolic = I.elimination_ideal((x0, x1, x2, x3, y0, y1, y2, y3))
parabolic = parabolic.change_ring(K[p01, p02, p03, p12, p13, p23])

print "Dimension: ", parabolic.dimension()-1
print "Degree: ", parabolic.hilbert_polynomial().coefficients()[1]
#print parabolic

### Hurwitz threefold, bitangent and principal congruences
We can use the same code used above to compute the Hurwitz three-fold (2) and the bitangent (2+2) and principal (3) congruences.

In [ ]:
R1.<x0, x1, x2, x3, p01, p02, p03, p12, p13, p23, t> = K[]

locus_type = '2' #change type

P = vector(R1,[1,0,-p12,-p13])
Q = vector(R1,[0, 1, p02, p03])
F1 = S_F.change_ring(R1)
poly = (F1.subs(x0=((P + t * Q)[0]), x1=((P + t * Q)[1]), 
                x2=((P + t * Q)[2]), x3 =((P + t * Q)[3]))).gens()[0]
c=poly.polynomial(t).coefficients()
disc=load("discriminants/disc%d_%s" %(degree,locus_type))
v = disc.ring().gens()
disc = disc.change_ring(K[v])

sub_dict = dict(zip((disc.ring()).gens(),c))
locus = (disc.subs(sub_dict)).change_ring(R1)
locus = ideal(locus)+ideal(1*p23-p02*p13+p03*p12) #.change_ring(R2)
locus = ideal(locus.groebner_basis())
locus = (locus.homogenize(p01)).change_ring(K[p01, p02, p03, p12, p13, p23])

print "Dimension: ", locus.dimension()-1
if locus_type == '2': # if threefold, we want degree of form
    print "Degree: ", locus.hilbert_polynomial().coefficients()[locus.dimension()-1]*factorial(
        locus.hilbert_polynomial().degree())/2
else:
    point = matrix(R2,[[1],[3],[5],[7]]) # random point and plane
    plane = matrix(R2,[[3],[7],[1],[6]])
    P = matrix(R1,[[0,p23,-p13,p12],[-p23,0,p03,-p02],[p13,-p03,0,p01],[-p12,p02,-p01,0]])
    Pdual = matrix(R1,[[0,-p01,-p02,-p03],[p01,0,-p12,-p13],[p02,p12,0,-p23],[p03,p13,p23,0]])
    point_inc = locus + ideal((P*point).list()).change_ring(K[p01, p02, p03, p12, p13, p23])
    plane_inc = locus + ideal((Pdual*plane).list()).change_ring(K[p01, p02, p03, p12, p13, p23])
    print "Order and class: ", (point_inc.hilbert_polynomial(), 
                                      plane_inc.hilbert_polynomial())

### Image contour
We project the surface from an arbitrary viewpoint.

In [ ]:
R2.<x0,x1,x2,x3,u0,u1,u2> = K['x0,x1,x2,x3,u0,u1,u2']

P = vector([1,2,0,1])
proj_matrix = matrix(K,[[-P[1],P[0],0,0],[-P[2],0,P[0],0],[-P[3],0,0,P[0]]])


X = matrix(R2, [[x0], [x1], [x2], [x3]])
U = matrix(R2, [[u0],[u1],[u2]])
S_F = S_F.change_ring(R2)

incidence_ideal = S_F + ideal((proj_matrix*X-U).list())
ramification = incidence_ideal + ideal(jacobian(incidence_ideal.gens(),(x0,x1,x2,x3)).minors(4))
outline = ramification.saturation(ideal(x0, x1, x2, x3))[0]
outline = outline.elimination_ideal((x0, x1, x2, x3))
outline = outline.change_ring(K[u0,u1,u2])

print outline

g = plot_curve(outline) #plot_curve(outline,'surf') to use singular
g.show()